**Author:** Bengü Lees  
**Final public leaderboard score:** 0.11879  
**Approximate rank:** 233  

This notebook contains my final workflow using Gradient Boosting, CatBoost and ElasticNet.  
I trained the models separately and combined only their final predictions using weighted blending.

In [ ]:
# House Prices Competition – Final Solution


# I load the training and test data here.
train_data = pd.read_csv("/content/train.csv")
test_data = pd.read_csv("/content/test.csv")


# I save the IDs from the test data because I need them later
# for the final Kaggle submission.
test_ids = test_data["Id"].copy()


# I separate the features from the value I want to predict.
# SalePrice is my target, so I remove it from X.
# I also remove Id because it is only a number used to identify each house.
X = train_data.drop(columns=["SalePrice", "Id"])
y = train_data["SalePrice"]


# I prepare the Kaggle test data in the same way
# and remove Id because the model should not learn from it.
X_kaggle_test = test_data.drop(columns=["Id"])


# I check the shapes to make sure everything looks correct.
print("Training features:", X.shape)
print("Target:", y.shape)
print("Kaggle test features:", X_kaggle_test.shape)
```


# I use 5-fold cross-validation so I can evaluate the models
# on different parts of the training data.
#
# The data is split into five parts.
# The model trains on four parts and validates on the remaining part.
# This is repeated until every part was used for validation once.
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# I transform SalePrice with log1p because the house prices
# are strongly right-skewed and Kaggle also evaluates the predictions
# using a logarithmic error.
y_log = np.log1p(y)



# I separate the numerical and categorical columns
# because they need different preprocessing.
numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude="number"
).columns.tolist()



# For the numerical columns, I replace missing values with the median.
# I use the median because it is less affected by extreme values.
gb_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)



# For categorical columns, I first replace missing values
# with the most common category.
#
# After that, I use one-hot encoding because the model
# cannot work directly with words such as "Normal" or "Excellent".
gb_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)



# I combine the preprocessing for the numerical
# and categorical features in one ColumnTransformer.
gb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            gb_numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            gb_categorical_transformer,
            categorical_features
        )
    ]
)



# Here I create my tuned Gradient Boosting model.
# These are the parameters that gave me the best result
# during my experiments.
gradient_boosting_model = GradientBoostingRegressor(
    n_estimators=1250,
    learning_rate=0.02,
    max_depth=3,
    min_samples_leaf=8,
    min_samples_split=10,
    subsample=0.8,
    loss="huber",
    random_state=42
)



# I put the preprocessing and the model into one pipeline.
# This makes sure the preprocessing is fitted only
# on the training part of each fold.
gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", gb_preprocessor),
        ("regressor", gradient_boosting_model)
    ]
)



# I create an empty array for the out-of-fold predictions.
# Every training house will later get a prediction from a model
# that did not train on that house.
gb_oof_predictions_log = np.zeros(len(X))


# I also create an empty array for the Kaggle test predictions.
# I will average the predictions from all five folds.
gb_test_predictions_log_cv = np.zeros(len(X_kaggle_test))



# I train Gradient Boosting five times,
# once for every cross-validation fold.
for fold_number, (train_indices, valid_indices) in enumerate(
    kf.split(X),
    start=1
):

    print(f"Training Gradient Boosting fold {fold_number} of 5...")


    # I select the training and validation rows for this fold.
    X_fold_train = X.iloc[train_indices]
    X_fold_valid = X.iloc[valid_indices]

    y_fold_train_log = y_log.iloc[train_indices]
    y_fold_valid_log = y_log.iloc[valid_indices]


    # I create a new copy of the pipeline for this fold.
    gb_fold_model = clone(gradient_boosting_pipeline)


    # I train the model only on the training part.
    gb_fold_model.fit(
        X_fold_train,
        y_fold_train_log
    )


    # I predict the validation part that the model has not seen before.
    gb_oof_predictions_log[valid_indices] = (
        gb_fold_model.predict(X_fold_valid)
    )


    # I also predict the Kaggle test data.
    # I divide by five because I want the average
    # of the five different fold predictions.
    gb_test_predictions_log_cv += (
        gb_fold_model.predict(X_kaggle_test) / 5
    )


    # I calculate the score for this fold
    # to see how well the model performed.
    fold_score = mean_squared_error(
        y_fold_valid_log,
        gb_oof_predictions_log[valid_indices]
    ) ** 0.5

    print(
        f"Fold {fold_number} Log RMSE: "
        f"{fold_score:.6f}"
    )



# CatBoost can work with categorical columns directly.
# I only need to replace the missing categories
# and convert all categorical values into strings.
X_catboost = X.copy()
X_kaggle_catboost = X_kaggle_test.copy()



# These are the CatBoost parameters that performed best
# during my extended parameter search.
catboost_fold_model = CatBoostRegressor(
    iterations=2619,
    depth=4,
    learning_rate=0.04,
    l2_leaf_reg=5,
    random_strength=2.0,
    bagging_temperature=1.0,
    border_count=64,
    grow_policy="SymmetricTree",
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)



# I check the numerical features for skewness.
# Some columns have many small or normal values
# but only a few extremely large values.
numeric_skewness = X[numeric_features].skew()


# I select the numerical columns that are strongly skewed.
# I only use columns without negative values
# because I want to apply log1p later.
skewed_numeric_features = [
    column
    for column in numeric_features
    if abs(numeric_skewness[column]) > 0.75
    and X[column].dropna().min() >= 0
]


# These are the remaining numerical columns
# that do not need the log transformation.
regular_numeric_features = [
    column
    for column in numeric_features
    if column not in skewed_numeric_features
]



# For the strongly skewed columns, I first fill missing values,
# then apply log1p and finally standardize the values.
#
# The log transformation makes extremely large values
# less dominant for ElasticNet.
elastic_skewed_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "log_transform",
            FunctionTransformer(
                np.log1p,
                feature_names_out="one-to-one"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)



# For the other numerical columns,
# I only fill missing values and standardize them.
elastic_regular_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)



# For categorical features, I replace missing values
# and then use one-hot encoding.
elastic_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)



# I use GridSearchCV to test different values
# for alpha and l1_ratio.
#
# Alpha controls how strong the regularization is.
# l1_ratio controls the balance between Lasso and Ridge regularization.
elastic_search = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid=elastic_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    n_jobs=-1,
    verbose=1
)



# I create out-of-fold predictions for ElasticNet.
# This means every training house is predicted
# by a model that did not train on that house.
elastic_oof_predictions_log = cross_val_predict(
    best_elastic_model,
    X,
    y_log,
    cv=kf,
    method="predict",
    n_jobs=-1
)



# I trained all three models separately.
# I did not combine the models during training.
#
# I only combine their final predictions here
# using the weights that gave me the best OOF result.
GB_WEIGHT = 0.2277
CATBOOST_WEIGHT = 0.4323
ELASTICNET_WEIGHT = 0.34



# I blend the three out-of-fold predictions in log space.
final_oof_predictions_log = (
    GB_WEIGHT * gb_oof_predictions_log
    + CATBOOST_WEIGHT
    * extended_catboost_oof_predictions_log
    + ELASTICNET_WEIGHT
    * elastic_oof_predictions_log
)



# I calculate the final ensemble score
# to compare it with the individual model scores.
final_oof_score = mean_squared_error(
    y_log,
    final_oof_predictions_log
) ** 0.5



# I combine the Kaggle test predictions
# with the same weights as the OOF predictions.
final_test_predictions_log = (
    GB_WEIGHT * gb_test_predictions_log_cv
    + CATBOOST_WEIGHT
    * extended_catboost_test_predictions_log_cv
    + ELASTICNET_WEIGHT
    * elastic_test_predictions_log_cv
)



# The models predicted the logarithm of SalePrice.
# Here I convert the predictions back
# into normal house-price values.
final_test_predictions = np.expm1(
    final_test_predictions_log
)



# Finally, I create the submission file
# with the original test IDs and my predicted SalePrice values.
submission = pd.DataFrame(
    {
        "Id": test_ids,
        "SalePrice": final_test_predictions
    }
)


# I save the final file without an extra index column
# because Kaggle only expects Id and SalePrice.
submission.to_csv(
    "final_house_prices_solution.csv",
    index=False
)

